In [1]:
import re
from typing import List, Dict, Optional
import numpy as np
import pandas as pd

In [2]:
def parse_perf_runs_from_cycles(log_text: str, cpu_hz: Optional[int] = None) -> List[Dict[str, float]]:
    """
    Parse logs with per-stage 'CPU TOTAL: <cycles> cycles' (e.g., Ethos-U prints).
    Returns: list of dicts, one per run, mapping stage -> seconds.

    - Detects CPU clock from: 'Processor internal clock: <Hz>Hz' unless cpu_hz is provided.
    - A run is segmented by 'INFO - Running inference on image'.
    - For each 'INFO - Profile for <Stage>:' block, the parser scans up to the next
      stage header (or end-of-run) and extracts the 'CPU TOTAL' value.
    """
    # Get CPU clock
    if cpu_hz is None:
        m = re.search(r"Processor internal clock:\s*(\d+)\s*Hz", log_text)
        if m:
            cpu_hz = int(m.group(1))
        else:
            raise ValueError("CPU frequency not found in log and cpu_hz not provided.")

    runs: List[Dict[str, float]] = []

    # Find run boundaries
    run_starts = [m.start() for m in re.finditer(r"INFO - Running inference on image", log_text)]
    if not run_starts:
        return runs
    run_starts.append(len(log_text))  # sentinel

    for i in range(len(run_starts) - 1):
        chunk = log_text[run_starts[i]: run_starts[i+1]]
        run_dict: Dict[str, float] = {}

        # Stage headers within this run
        stage_iter = list(re.finditer(r"INFO - Profile for\s+([^:]+):", chunk))
        if not stage_iter:
            continue

        # End positions to slice each stage block
        ends = [m.start() for m in stage_iter[1:]] + [len(chunk)]

        for (m, end_pos) in zip(stage_iter, ends):
            stage = " ".join(m.group(1).split())
            block = chunk[m.end():end_pos]

            # Extract CPU TOTAL within this stage block
            cpu_match = re.search(r"CPU TOTAL:\s*(\d+)\s*cycles", block)
            if cpu_match:
                cycles = int(cpu_match.group(1))
                run_dict[stage] = cycles / float(cpu_hz) * 1000  # milliseconds

        if run_dict:
            runs.append(run_dict)

    return runs


def parse_perf_runs_native(log_text: str) -> List[Dict[str, float]]:
    """
    Parse logs and return a list of dicts (one per run), mapping stage -> seconds.
    A run starts at: 'INFO - Running inference on image ...'
    Stages are detected by:
      'INFO - Profile for <Stage>:'
      (followed later by) 'INFO - Duration: <num> microseconds'
    """
    runs: List[Dict[str, float]] = []

    # Find run boundaries
    run_starts = [m.start() for m in re.finditer(r"INFO - Running inference on image", log_text)]
    if not run_starts:
        return runs
    run_starts.append(len(log_text))  # sentinel

    for i in range(len(run_starts) - 1):
        chunk = log_text[run_starts[i]: run_starts[i+1]]
        run_dict: Dict[str, float] = {}

        # Extract (stage, duration_us) pairs inside this run
        for stage, us in re.findall(
            r"INFO - Profile for\s+([^:]+):.*?INFO - Duration:\s+(\d+)\s+microseconds",
            chunk,
            flags=re.DOTALL,
        ):
            stage = " ".join(stage.split())  # tidy spacing
            run_dict[stage] = int(us) / 1_000.0  # milliseconds

        if run_dict:
            runs.append(run_dict)

    return runs


def summarize_perf(runs: List[Dict[str, float]]) -> Dict[str, Dict[str, float]]:
    """
    Given output of parse_perf_runs, compute mean & std for each stage using numpy.
    Returns dict: {stage: {"mean": float, "std": float}}
    """
    stage_values: Dict[str, List[float]] = {}
    for run in runs:
        for stage, val in run.items():
            stage_values.setdefault(stage, []).append(val)

    summary: Dict[str, Dict[str, float]] = {}
    for stage, vals in stage_values.items():
        arr = np.array(vals, dtype=float)
        summary[stage] = {
            "mean": float(np.mean(arr)),
            "std": float(np.std(arr, ddof=1)) if len(arr) > 1 else 0.0
        }
    return summary


def parse_perf_runs_from_cycles_v2(log_text: str, cpu_hz: Optional[int] = None) -> List[Dict[str, float]]:
    """
    Parse logs with per-stage 'CPU TOTAL: <cycles>' including stages that come
    BEFORE 'Running inference...' within the same user Choice segment.

    Run segmentation:
      - Prefer segments between consecutive lines matching '^\\s*Choice:\\s*\\d*'.
      - If no 'Choice:' prompts exist, fall back to segments starting at
        'INFO - Running inference on image'.

    Units: returns seconds for each stage.
    """
    # Detect CPU frequency if not provided
    if cpu_hz is None:
        m = re.search(r"Processor internal clock:\s*(\d+)\s*Hz", log_text)
        if m:
            cpu_hz = int(m.group(1))
        else:
            raise ValueError("CPU frequency not found in log and cpu_hz not provided.")

    def parse_chunk(chunk: str) -> Dict[str, float]:
        out: Dict[str, float] = {}
        # Match stage headers; colon is optional in some lines
        stage_matches = list(re.finditer(r"INFO - Profile for\s+([^:\n]+):?", chunk))
        if not stage_matches:
            return out
        # Block = region from this header to the next header (or end of chunk)
        ends = [m.start() for m in stage_matches[1:]] + [len(chunk)]
        for m, end_pos in zip(stage_matches, ends):
            stage = " ".join(m.group(1).split())
            block = chunk[m.end():end_pos]
            cpu_match = re.search(r"CPU TOTAL:\s*(\d+)\s*cycles", block)
            if cpu_match:
                cycles = int(cpu_match.group(1))
                out[stage] = cycles / float(cpu_hz) * 1000  # milliseconds
        return out

    runs: List[Dict[str, float]] = []

    # Primary: segment by Choice prompts
    choice_starts = [m.span()[0] for m in re.finditer(r"^\s*Choice:\s*\d*", log_text, flags=re.MULTILINE)]
    if choice_starts:
        choice_starts.append(len(log_text))
        for i in range(len(choice_starts) - 1):
            chunk = log_text[choice_starts[i]:choice_starts[i+1]]
            run_dict = parse_chunk(chunk)
            if run_dict:
                runs.append(run_dict)
        return runs

    # Fallback: segment by 'Running inference on image'
    run_starts = [m.start() for m in re.finditer(r"INFO - Running inference on image", log_text)]
    if not run_starts:
        return runs
    run_starts.append(len(log_text))
    for i in range(len(run_starts) - 1):
        chunk = log_text[run_starts[i]:run_starts[i+1]]
        run_dict = parse_chunk(chunk)
        if run_dict:
            runs.append(run_dict)
    return runs


def add_prefix_sums(
    runs: List[Dict[str, float]], 
    prefix: str = "get_image_data", 
    sum_key: str = "get_image_data_total"
) -> List[Dict[str, float]]:
    """
    For each run dict, compute the sum of all stage values whose key starts
    with `prefix` and store it under `sum_key`.

    Args:
        runs: list of dicts from parse_perf_runs_from_cycles_v2
        prefix: stage name prefix to sum over
        sum_key: key name to insert into each run dict with the sum
    
    Returns:
        new list of dicts with the extra aggregated value
    """
    out_runs: List[Dict[str, float]] = []
    for run in runs:
        total = sum(val for k, val in run.items() if k.startswith(prefix))
        new_run = dict(run)
        if total > 0:
            new_run[sum_key] = total
        out_runs.append(new_run)
    return out_runs

def add_prefix_and_other_sums(
    runs: List[Dict[str, float]], 
    prefix: str = "get_image_data", 
    prefix_sum_key: str = "get_image_data_total",
    other_sum_key: str = "other_stages_total"
) -> List[Dict[str, float]]:
    """
    For each run dict, compute:
      - sum of all stage values whose key starts with `prefix`
      - sum of all remaining stage values
    and store them under `prefix_sum_key` and `other_sum_key`.

    Args:
        runs: list of dicts from parse_perf_runs_from_cycles_v2
        prefix: stage name prefix to group
        prefix_sum_key: key name to store sum of prefix-matching stages
        other_sum_key: key name to store sum of all other stages
    
    Returns:
        new list of dicts with the extra aggregated values
    """
    out_runs: List[Dict[str, float]] = []
    for run in runs:
        prefix_total = sum(val for k, val in run.items() if k.startswith(prefix))
        other_total = sum(val for k, val in run.items() if not k.startswith(prefix))
        
        new_run = dict(run)
        if prefix_total > 0:
            new_run[prefix_sum_key] = prefix_total
        if other_total > 0:
            new_run[other_sum_key] = other_total
        out_runs.append(new_run)
    return out_runs

In [3]:
logs_native = """INFO - ARM ML Embedded Evaluation Kit
INFO - Version 23.08.0 Build date: Aug 20 2025 @ 11:39:16
INFO - Copyright 2021-2023 Arm Limited and/or its affiliates <open-source-office@arm.com>

INFO - Added  support to op resolver
INFO - Creating allocator using tensor arena at 0x6094b9e8c030
INFO - Allocating tensors


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  Choice: 1

INFO - Running inference on image 0 => img_320.bmp
Final results:
Total number of inferences: 4
Box: [183.85 30.64 61.28 61.28]. Position: [204.91 82.35]. Confidence: 0.80
Box: [183.85 91.92 91.92 30.64]. Position: [204.91 122.57]. Confidence: 0.80
Box: [122.57 153.21 61.28 91.92]. Position: [164.70 143.63]. Confidence: 0.80
Box: [91.92 0.00 61.28 91.92]. Position: [143.63 82.35]. Confidence: 0.61
INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 168136 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 4363916 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13214 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6086563 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8331120 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4185715 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3549425 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 1521482356 cycles
"""

In [4]:
logs_native_nano = """INFO - ARM ML Embedded Evaluation Kit
INFO - Version 23.08.0 Build date: Aug 20 2025 @ 18:05:26
INFO - Copyright 2021-2023 Arm Limited and/or its affiliates <open-source-office@arm.com>

INFO - Added  support to op resolver
INFO - Creating allocator using tensor arena at 0x5c6b010c6030
INFO - Allocating tensors


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  Choice: 1

INFO - Running inference on image 0 => img_320.bmp
Final results:
Total number of inferences: 5
Box: [175.97 75.88 101.71 50.05]. Position: [224.40 101.71]. Confidence: 0.86
Box: [75.88 0.00 75.88 101.71]. Position: [142.07 80.72]. Confidence: 0.80
Box: [125.92 125.92 50.05 101.71]. Position: [163.05 163.05]. Confidence: 0.80
Box: [175.97 25.83 50.05 75.88]. Position: [203.41 61.35]. Confidence: 0.50
Box: [50.05 175.97 50.05 50.05]. Position: [101.71 203.41]. Confidence: 0.50
INFO - Profile for CopyOutput:
INFO - Duration: 59 microseconds
INFO - Profile for Inference:
INFO - Duration: 803755 microseconds
INFO - Profile for ModelInit:
INFO - Duration: 1400 microseconds
INFO - Profile for PostProcessing:
INFO - Duration: 11 microseconds
INFO - Profile for PreProcessing:
INFO - Duration: 54 microseconds


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  Choice: 1

INFO - Running inference on image 0 => img_320.bmp
Final results:
Total number of inferences: 6
Box: [175.97 75.88 101.71 50.05]. Position: [224.40 122.69]. Confidence: 0.86
Box: [75.88 0.00 75.88 101.71]. Position: [142.07 80.72]. Confidence: 0.80
Box: [125.92 125.92 50.05 101.71]. Position: [163.05 163.05]. Confidence: 0.80
Box: [175.97 25.83 50.05 75.88]. Position: [203.41 61.35]. Confidence: 0.61
Box: [318.03 279.29 0.00 38.75]. Position: [318.03 297.05]. Confidence: 0.39
Box: [140.45 177.58 38.75 62.96]. Position: [174.35 184.04]. Confidence: 0.29
INFO - Profile for CopyOutput:
INFO - Duration: 27 microseconds
INFO - Profile for Inference:
INFO - Duration: 790995 microseconds
INFO - Profile for PostProcessing:
INFO - Duration: 10 microseconds
INFO - Profile for PreProcessing:
INFO - Duration: 43 microseconds


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  Choice: 1

INFO - Running inference on image 0 => img_320.bmp
Final results:
Total number of inferences: 5
Box: [175.97 75.88 101.71 50.05]. Position: [224.40 122.69]. Confidence: 0.86
Box: [75.88 0.00 75.88 101.71]. Position: [142.07 80.72]. Confidence: 0.80
Box: [125.92 125.92 50.05 101.71]. Position: [163.05 163.05]. Confidence: 0.80
Box: [175.97 25.83 50.05 75.88]. Position: [203.41 61.35]. Confidence: 0.61
Box: [140.45 177.58 38.75 62.96]. Position: [174.35 184.04]. Confidence: 0.29
INFO - Profile for CopyOutput:
INFO - Duration: 25 microseconds
INFO - Profile for Inference:
INFO - Duration: 797537 microseconds
INFO - Profile for PostProcessing:
INFO - Duration: 10 microseconds
INFO - Profile for PreProcessing:
INFO - Duration: 25 microseconds


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  Choice: """
  
logs_alif_nano = """[sudo] password for pglab3: 
picocom v3.1

port is        : /dev/ttyUSB0
flowcontrol    : none
baudrate is    : 921600
parity is      : none
databits are   : 8
stopbits are   : 1
escape is      : C-a
local echo is  : no
noinit is      : no
noreset is     : no
hangup is      : no
nolock is      : no
send_cmd is    : sz -vv
receive_cmd is : rz -vv -E
imap is        : 
omap is        : 
emap is        : crcrlf,delbs,
logfile is     : none
initstring     : none
exit_after is  : not set
exit is        : no

Type [C-a] [C-h] to see available commands
Terminal ready

Wakeup Reason (LPTIMER) Interrupt!

Reset Reason [0](UNKNOWN)
PointGrab Board Rev.A
WARN - Backup 4K SRAM is not valid -> initializing!
INFO - Processor internal clock: 160000000Hz
INFO - platform_init: complete
INFO - Ethos-U device initialised
INFO - Ethos-U version info:
INFO -  Arch:       v1.1.0
INFO -  Driver:     v0.16.0
INFO -  MACs/cc:    128
INFO -  Cmd stream: v0
INFO - Target system design: Ensemble
INFO - ARM ML Embedded Evaluation Kit
INFO - Version 23.08.0 Build date: Aug 20 2025 @ 18:12:08
INFO - Copyright 2021-2023 Arm Limited and/or its affiliates <open-source-office@arm.com>

INFO - Added ethos-u support to op resolver
INFO - Creating allocator using tensor arena at 0x2000000
INFO - Allocating tensors


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

dump binary memory /tmp/cam_image0_Bayer_to_RGB_320p.tif 0x8083400 0x80BB8B0 
INFO - Running inference on image from camera
Final results:
              Total number of inferences: 4
                                           Box: [503.69 151.75 151.75 151.75]. Position: [529.52 203.41]. Confidence: 0.80
                                                                                                                          Box: [303.50 0.00 100.09 100.09]. Position: [364.85 80.72]. Confidence: 0.61
   Box: [503.69 303.50 151.75 51.66]. Position: [529.52 326.10]. Confidence: 0.61
                                                                                 Box: [100.09 303.50 100.09 151.75]. Position: [161.44 326.10]. Confidence: 0.61
                                                                                                                                                                INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 230032 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 29387472 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2247 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8677430 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4407659 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 21135652 cycles
INFO - NPU TOTAL: 21137899 cycles
INFO - CPU TOTAL: 25117230 cycles
INFO - Profile for ModelInit:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 567606 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 91699 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 135927 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

dump binary memory /tmp/cam_image0_Bayer_to_RGB_320p.tif 0x8083400 0x80BB8B0 
INFO - Running inference on image from camera
Final results:
              Total number of inferences: 4
                                           Box: [503.69 151.75 151.75 151.75]. Position: [529.52 203.41]. Confidence: 0.80
                                                                                                                          Box: [100.09 303.50 100.09 151.75]. Position: [161.44 326.10]. Confidence: 0.71
      Box: [303.50 0.00 100.09 100.09]. Position: [364.85 80.72]. Confidence: 0.61
                                                                                  Box: [503.69 303.50 151.75 51.66]. Position: [529.52 326.10]. Confidence: 0.61
                                                                                                                                                                INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 229227 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 25780418 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2265 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8677430 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4407659 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 21134526 cycles
INFO - NPU TOTAL: 21136791 cycles
INFO - CPU TOTAL: 25114179 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 89348 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 135872 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

dump binary memory /tmp/cam_image0_Bayer_to_RGB_320p.tif 0x8083400 0x80BB8B0 
INFO - Running inference on image from camera
Final results:
              Total number of inferences: 4
                                           Box: [503.69 151.75 151.75 151.75]. Position: [529.52 203.41]. Confidence: 0.80
                                                                                                                          Box: [303.50 0.00 100.09 100.09]. Position: [364.85 80.72]. Confidence: 0.61
   Box: [503.69 303.50 151.75 51.66]. Position: [529.52 326.10]. Confidence: 0.61
                                                                                 Box: [100.09 303.50 100.09 151.75]. Position: [161.44 326.10]. Confidence: 0.61
                                                                                                                                                                INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 228665 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 26845430 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2336 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8677430 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4407659 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 21135894 cycles
INFO - NPU TOTAL: 21138230 cycles
INFO - CPU TOTAL: 25115409 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 92490 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 130142 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: """

In [5]:
summary = summarize_perf(parse_perf_runs_native(logs_native))
pd.DataFrame.from_dict(summary, orient='index').reset_index().rename(columns={'index': 'Stage'}).round(3)

,Stage


In [6]:
summary = summarize_perf(parse_perf_runs_from_cycles(logs_alif))
pd.DataFrame.from_dict(summary, orient='index').reset_index().rename(columns={'index': 'Stage'}).round(3)

NameError: name 'logs_alif' is not defined

In [7]:
summary = summarize_perf(parse_perf_runs_native(logs_native_nano))
pd.DataFrame.from_dict(summary, orient='index').reset_index().rename(columns={'index': 'Stage'}).round(3)

,Stage,mean,std
0,CopyOutput,0.037,0.019
1,Inference,797.429,6.381
2,ModelInit,1.400,0.000
3,PostProcessing,0.010,0.001
4,PreProcessing,0.041,0.015


In [8]:
summary = summarize_perf(parse_perf_runs_from_cycles(logs_alif_nano))
pd.DataFrame.from_dict(summary, orient='index').reset_index().rename(columns={'index': 'Stage'}).round(3)

,Stage,mean,std
0,CopyOutput,1.433,0.004
1,GetImage,170.861,11.583
2,Inference,156.973,0.010
3,ModelInit,3.548,0.000
4,PostProcessing,0.570,0.010
5,PreProcessing,0.837,0.021


In [9]:
logs_extended = """[sudo] password for pglab3: 
picocom v3.1

port is        : /dev/ttyUSB0
flowcontrol    : none
baudrate is    : 921600
parity is      : none
databits are   : 8
stopbits are   : 1
escape is      : C-a
local echo is  : no
noinit is      : no
noreset is     : no
hangup is      : no
nolock is      : no
send_cmd is    : sz -vv
receive_cmd is : rz -vv -E
imap is        : 
omap is        : 
emap is        : crcrlf,delbs,
logfile is     : none
initstring     : none
exit_after is  : not set
exit is        : no

Type [C-a] [C-h] to see available commands
Terminal ready

Wakeup Reason (LPTIMER) Interrupt!

Reset Reason [0](UNKNOWN)
PointGrab Board Rev.A
WARN - Backup 4K SRAM is not valid -> initializing!
INFO - Processor internal clock: 160000000Hz
INFO - platform_init: complete
INFO - Ethos-U device initialised
INFO - Ethos-U version info:
INFO -  Arch:       v1.1.0
INFO -  Driver:     v0.16.0
INFO -  MACs/cc:    128
INFO -  Cmd stream: v0
INFO - Target system design: Ensemble
INFO - ARM ML Embedded Evaluation Kit
INFO - Version 23.08.0 Build date: Aug 21 2025 @ 12:10:53
INFO - Copyright 2021-2023 Arm Limited and/or its affiliates <open-source-office@arm.com>

INFO - Added ethos-u support to op resolver
INFO - Creating allocator using tensor arena at 0x2000000
INFO - Allocating tensors


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 165068 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 6353383 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13885 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6185311 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8402478 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4216384 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3723568 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 1168 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 230262 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 30228076 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2247 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20295791 cycles
INFO - NPU TOTAL: 20298038 cycles
INFO - CPU TOTAL: 24278602 cycles
INFO - Profile for ModelInit:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 567902 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 77878 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 129951 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 164536 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 4343484 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 14028 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6179005 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8390838 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4215628 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3596789 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 1108 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 229014 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 28070909 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2340 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20295818 cycles
INFO - NPU TOTAL: 20298158 cycles
INFO - CPU TOTAL: 24277391 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 60617 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 131811 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 164954 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 3573684 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13938 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6176708 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8391240 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4216008 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3598221 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 978 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 228823 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 27301343 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2227 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20296463 cycles
INFO - NPU TOTAL: 20298690 cycles
INFO - CPU TOTAL: 24276945 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 76844 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 130834 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 164963 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 4263606 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13906 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6180845 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8400327 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4215284 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3613964 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 934 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 228862 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 28019098 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2177 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20295889 cycles
INFO - NPU TOTAL: 20298066 cycles
INFO - CPU TOTAL: 24276718 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 68252 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 135010 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 164818 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 2401528 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13845 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6180471 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8401208 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4215502 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3609331 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 948 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 228985 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 26153213 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2287 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20295812 cycles
INFO - NPU TOTAL: 20298099 cycles
INFO - CPU TOTAL: 24276714 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 60609 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 128524 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 164396 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 3529469 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13899 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6179333 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8400788 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4215376 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3598537 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 961 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 229001 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 27268358 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2341 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20295880 cycles
INFO - NPU TOTAL: 20298221 cycles
INFO - CPU TOTAL: 24276283 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 69425 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 135231 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 167915 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 4885687 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13999 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6180423 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8406008 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4215611 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3603104 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 1039 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 228877 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 28639206 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2255 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20295646 cycles
INFO - NPU TOTAL: 20297901 cycles
INFO - CPU TOTAL: 24276843 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 66216 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 135513 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 167666 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 2717896 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13812 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6178415 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8397080 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4215832 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3596825 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 995 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 229048 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 26453857 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2324 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20296094 cycles
INFO - NPU TOTAL: 20298418 cycles
INFO - CPU TOTAL: 24277438 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 60350 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 129562 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 167251 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 4764166 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13911 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6183687 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8393970 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4215718 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3600076 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 939 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 228948 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 28505206 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2269 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20295600 cycles
INFO - NPU TOTAL: 20297869 cycles
INFO - CPU TOTAL: 24276432 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 73344 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 135188 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 165902 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 4116550 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13906 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6179288 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8402591 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4214859 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3609114 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 939 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 228886 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 27868642 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2285 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20295422 cycles
INFO - NPU TOTAL: 20297707 cycles
INFO - CPU TOTAL: 24276412 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 75355 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 129834 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: """

In [10]:
summary = summarize_perf(parse_perf_runs_from_cycles_v2(logs_extended))
pd.DataFrame.from_dict(summary, orient='index').reset_index().rename(columns={'index': 'Stage'}).round(3)

,Stage,mean,std
0,get_image_data.camera_start,1.036,0.008
1,get_image_data.camera_wait,25.593,7.102
2,get_image_data.RTSS_InvalidateDCache_by_Addr,0.087,0.000
3,get_image_data.process_autogain_upd,38.627,0.016
4,get_image_data.dc1394_bayer_Simple,52.492,0.032
5,get_image_data.interpolate,26.348,0.003
6,get_image_data.white_balance,22.593,0.241
7,get_image_data.apply_clahe,0.006,0.001
8,CopyOutput,1.432,0.003
9,GetImage,174.067,7.292


In [11]:
summary = summarize_perf(add_prefix_and_other_sums(parse_perf_runs_from_cycles_v2(logs_extended)))
pd.DataFrame.from_dict(summary, orient='index').reset_index().rename(columns={'index': 'Stage'}).round(3)

,Stage,mean,std
0,get_image_data.camera_start,1.036,0.008
1,get_image_data.camera_wait,25.593,7.102
2,get_image_data.RTSS_InvalidateDCache_by_Addr,0.087,0.000
3,get_image_data.process_autogain_upd,38.627,0.016
4,get_image_data.dc1394_bayer_Simple,52.492,0.032
5,get_image_data.interpolate,26.348,0.003
6,get_image_data.white_balance,22.593,0.241
7,get_image_data.apply_clahe,0.006,0.001
8,CopyOutput,1.432,0.003
9,GetImage,174.067,7.292


In [14]:
logs_baseline = """picocom v3.1

port is        : /dev/ttyUSB0
flowcontrol    : none
baudrate is    : 921600
parity is      : none
databits are   : 8
stopbits are   : 1
escape is      : C-a
local echo is  : no
noinit is      : no
noreset is     : no
hangup is      : no
nolock is      : no
send_cmd is    : sz -vv
receive_cmd is : rz -vv -E
imap is        : 
omap is        : 
emap is        : crcrlf,delbs,
logfile is     : none
initstring     : none
exit_after is  : not set
exit is        : no

Type [C-a] [C-h] to see available commands
Terminal ready

Reset Reason [0](UNKNOWN)
PointGrab Board Rev.A
WARN - Backup 4K SRAM is not valid -> initializing!
INFO - Processor internal clock: 160000000Hz
INFO - platform_init: complete
INFO - Ethos-U device initialised
INFO - Ethos-U version info:
INFO - 	Arch:       v1.1.0
INFO - 	Driver:     v0.16.0
INFO - 	MACs/cc:    128
INFO - 	Cmd stream: v0
INFO - Target system design: Ensemble
INFO - ARM ML Embedded Evaluation Kit
INFO - Version 23.08.0 Build date: Sep 18 2025 @ 12:39:12
INFO - Copyright 2021-2023 Arm Limited and/or its affiliates <open-source-office@arm.com>

INFO - Added ethos-u support to op resolver
INFO - Creating allocator using tensor arena at 0x2000000
INFO - Allocating tensors


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 165583 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 6361154 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13175 cycles
INFO - Profile for process_autogain_upd.calculate_pixels_counts
INFO - CPU TOTAL: 6160602 cycles
INFO - Profile for process_autogain_upd.camera_gain_get
INFO - CPU TOTAL: 669 cycles
INFO - Profile for process_autogain_upd.camera_gain_update
INFO - CPU TOTAL: 487 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6630746 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8399016 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4216401 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3732174 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 1194 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 229222 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 30745098 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2364 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20303999 cycles
INFO - NPU TOTAL: 20306363 cycles
INFO - CPU TOTAL: 24289088 cycles
INFO - Profile for ModelInit:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 568420 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 100566 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 135119 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 167980 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 4238848 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13238 cycles
INFO - Profile for process_autogain_upd.calculate_pixels_counts
INFO - CPU TOTAL: 6160308 cycles
INFO - Profile for process_autogain_upd.camera_gain_get
INFO - CPU TOTAL: 535 cycles
INFO - Profile for process_autogain_upd.camera_gain_update
INFO - CPU TOTAL: 128 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6630445 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8403493 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4216261 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3610598 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 1141 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 229519 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 28506610 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2190 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20304405 cycles
INFO - NPU TOTAL: 20306595 cycles
INFO - CPU TOTAL: 24288603 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 96640 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 130820 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 165979 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 2881881 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13240 cycles
INFO - Profile for process_autogain_upd.calculate_pixels_counts
INFO - CPU TOTAL: 6159266 cycles
INFO - Profile for process_autogain_upd.camera_gain_get
INFO - CPU TOTAL: 530 cycles
INFO - Profile for process_autogain_upd.camera_gain_update
INFO - CPU TOTAL: 99 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6629299 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8397603 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4215152 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3610933 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 1052 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 228876 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 27137665 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2130 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20303501 cycles
INFO - NPU TOTAL: 20305631 cycles
INFO - CPU TOTAL: 24287102 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 96382 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 135356 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 164088 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 3555864 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13309 cycles
INFO - Profile for process_autogain_upd.calculate_pixels_counts
INFO - CPU TOTAL: 6159748 cycles
INFO - Profile for process_autogain_upd.camera_gain_get
INFO - CPU TOTAL: 513 cycles
INFO - Profile for process_autogain_upd.camera_gain_update
INFO - CPU TOTAL: 130 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6629863 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8399554 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4215722 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3605102 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 1018 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 228926 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 27808966 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2420 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20304013 cycles
INFO - NPU TOTAL: 20306433 cycles
INFO - CPU TOTAL: 24287883 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 96341 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 135328 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 164643 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 4355582 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13205 cycles
INFO - Profile for process_autogain_upd.calculate_pixels_counts
INFO - CPU TOTAL: 6159455 cycles
INFO - Profile for process_autogain_upd.camera_gain_get
INFO - CPU TOTAL: 537 cycles
INFO - Profile for process_autogain_upd.camera_gain_update
INFO - CPU TOTAL: 127 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6629511 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8402314 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4215633 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3598391 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 1022 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 228950 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 28604564 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2410 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20303351 cycles
INFO - NPU TOTAL: 20305761 cycles
INFO - CPU TOTAL: 24287216 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 97624 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 135423 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI


"""

In [16]:
logs_npu_upd = """picocom v3.1

port is        : /dev/ttyUSB0
flowcontrol    : none
baudrate is    : 921600
parity is      : none
databits are   : 8
stopbits are   : 1
escape is      : C-a
local echo is  : no
noinit is      : no
noreset is     : no
hangup is      : no
nolock is      : no
send_cmd is    : sz -vv
receive_cmd is : rz -vv -E
imap is        : 
omap is        : 
emap is        : crcrlf,delbs,
logfile is     : none
initstring     : none
exit_after is  : not set
exit is        : no

Type [C-a] [C-h] to see available commands
Terminal ready

Reset Reason [0](UNKNOWN)
PointGrab Board Rev.A
WARN - Backup 4K SRAM is not valid -> initializing!
INFO - Processor internal clock: 160000000Hz
INFO - platform_init: complete
INFO - Ethos-U device initialised
INFO - Ethos-U version info:
INFO - 	Arch:       v1.1.0
INFO - 	Driver:     v0.16.0
INFO - 	MACs/cc:    128
INFO - 	Cmd stream: v0
INFO - Target system design: Ensemble
INFO - ARM ML Embedded Evaluation Kit
INFO - Version 23.08.0 Build date: Sep 18 2025 @ 12:56:30
INFO - Copyright 2021-2023 Arm Limited and/or its affiliates <open-source-office@arm.com>

INFO - Added ethos-u support to op resolver
INFO - Creating allocator using tensor arena at 0x2000000
INFO - Allocating tensors


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 167978 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 6350858 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13344 cycles
INFO - Profile for process_autogain_upd.calculate_pixels_counts
INFO - CPU TOTAL: 6172518 cycles
INFO - Profile for process_autogain_upd.camera_gain_get
INFO - CPU TOTAL: 644 cycles
INFO - Profile for process_autogain_upd.camera_gain_update
INFO - CPU TOTAL: 488 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6642666 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8405719 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4216597 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3727645 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 1257 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 230176 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 30751683 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 1104 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8288508 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4205247 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 19574848 cycles
INFO - NPU TOTAL: 19575952 cycles
INFO - CPU TOTAL: 19598748 cycles
INFO - Profile for ModelInit:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 450865 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 110848 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 136284 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 166772 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 3806668 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13206 cycles
INFO - Profile for process_autogain_upd.calculate_pixels_counts
INFO - CPU TOTAL: 6170433 cycles
INFO - Profile for process_autogain_upd.camera_gain_get
INFO - CPU TOTAL: 553 cycles
INFO - Profile for process_autogain_upd.camera_gain_update
INFO - CPU TOTAL: 99 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6640556 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8404081 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4216187 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3601309 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 1332 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 229146 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 28072799 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 950 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8288508 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4205247 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 19575083 cycles
INFO - NPU TOTAL: 19576033 cycles
INFO - CPU TOTAL: 19597586 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 97919 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 135872 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 164383 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 3971570 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13148 cycles
INFO - Profile for process_autogain_upd.calculate_pixels_counts
INFO - CPU TOTAL: 6171521 cycles
INFO - Profile for process_autogain_upd.camera_gain_get
INFO - CPU TOTAL: 527 cycles
INFO - Profile for process_autogain_upd.camera_gain_update
INFO - CPU TOTAL: 99 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6641632 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8404252 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4216165 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3600994 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 1196 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 228844 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 28235731 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 895 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8288508 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4205247 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 19575309 cycles
INFO - NPU TOTAL: 19576204 cycles
INFO - CPU TOTAL: 19597298 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 97500 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 134879 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 167783 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 2445522 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13277 cycles
INFO - Profile for process_autogain_upd.calculate_pixels_counts
INFO - CPU TOTAL: 6172514 cycles
INFO - Profile for process_autogain_upd.camera_gain_get
INFO - CPU TOTAL: 508 cycles
INFO - Profile for process_autogain_upd.camera_gain_update
INFO - CPU TOTAL: 97 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6642582 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8396796 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4216149 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3609546 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 1129 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 228987 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 26715472 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 902 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8288508 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4205247 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 19575787 cycles
INFO - NPU TOTAL: 19576689 cycles
INFO - CPU TOTAL: 19597493 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 98862 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 135605 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

INFO - Profile for get_image_data.camera_start
INFO - CPU TOTAL: 165738 cycles
INFO - Profile for get_image_data.camera_wait
INFO - CPU TOTAL: 2426435 cycles
INFO - Profile for get_image_data.RTSS_InvalidateDCache_by_Addr
INFO - CPU TOTAL: 13150 cycles
INFO - Profile for process_autogain_upd.calculate_pixels_counts
INFO - CPU TOTAL: 6172805 cycles
INFO - Profile for process_autogain_upd.camera_gain_get
INFO - CPU TOTAL: 588 cycles
INFO - Profile for process_autogain_upd.camera_gain_update
INFO - CPU TOTAL: 98 cycles
INFO - Profile for get_image_data.process_autogain_upd
INFO - CPU TOTAL: 6642895 cycles
INFO - Profile for get_image_data.dc1394_bayer_Simple
INFO - CPU TOTAL: 8410217 cycles
INFO - Profile for get_image_data.interpolate
INFO - CPU TOTAL: 4215562 cycles
INFO - Profile for get_image_data.white_balance
INFO - CPU TOTAL: 3599182 cycles
INFO - Profile for get_image_data.apply_clahe
INFO - CPU TOTAL: 1108 cycles
INFO - Running inference on image from camera
INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 229092 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 26696875 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 927 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8288508 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4205247 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 19575124 cycles
INFO - NPU TOTAL: 19576051 cycles
INFO - CPU TOTAL: 19597168 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 97981 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 135651 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 

"""

In [17]:
summary = summarize_perf(add_prefix_and_other_sums(parse_perf_runs_from_cycles_v2(logs_baseline)))
pd.DataFrame.from_dict(summary, orient='index').reset_index().rename(columns={'index': 'Stage'}).round(3)

,Stage,mean,std
0,get_image_data.camera_start,1.035,0.009
1,get_image_data.camera_wait,26.742,8.160
2,get_image_data.RTSS_InvalidateDCache_by_Addr,0.083,0.000
3,process_autogain_upd.calculate_pixels_counts,38.499,0.004
4,process_autogain_upd.camera_gain_get,0.003,0.000
5,process_autogain_upd.camera_gain_update,0.001,0.001
6,get_image_data.process_autogain_upd,41.437,0.004
7,get_image_data.dc1394_bayer_Simple,52.502,0.015
8,get_image_data.interpolate,26.349,0.003
9,get_image_data.white_balance,22.696,0.353


In [18]:
summary = summarize_perf(add_prefix_and_other_sums(parse_perf_runs_from_cycles_v2(logs_npu_upd)))
pd.DataFrame.from_dict(summary, orient='index').reset_index().rename(columns={'index': 'Stage'}).round(3)

,Stage,mean,std
0,get_image_data.camera_start,1.041,0.009
1,get_image_data.camera_wait,23.751,10.009
2,get_image_data.RTSS_InvalidateDCache_by_Addr,0.083,0.001
3,process_autogain_upd.calculate_pixels_counts,38.575,0.006
4,process_autogain_upd.camera_gain_get,0.004,0.000
5,process_autogain_upd.camera_gain_update,0.001,0.001
6,get_image_data.process_autogain_upd,41.513,0.006
7,get_image_data.dc1394_bayer_Simple,52.526,0.030
8,get_image_data.interpolate,26.351,0.002
9,get_image_data.white_balance,22.673,0.350
